In [21]:
%load_ext autoreload
%autoreload 2

import open3d as o3d
import numpy as np
import os
import sys
from glob import glob
import pickle

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')
from utils.write_output import write_array2txt

from cylinder_extraction.cylinder import Cylinder
from cylinder_extraction.station import Station
from registration.utils import multiway_load_point_clouds as load_point_clouds
from registration.utils import multiway_full_registration_with_sequence as full_registration
from registration.utils import pairwise_registration, combine_point_cloud, multiway_filter_point_clouds

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Input
The first part of the tutorial code reads three point clouds from files. The point clouds are downsampled and visualized together. They are misaligned.

In [22]:
phase = 'p2l'

In [23]:
voxel_size = 0.02
data_dir = 'D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice'
if phase == 'p2p':
    data_dir = os.path.join(data_dir, 'multiway')
# pcds_down, pcds_full = load_point_clouds(data_dir,voxel_size)

In [24]:
root_dir = 'D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica'
tank_name = '3'
with open(os.path.join(root_dir, 'tank_info.pkl'), 'rb') as f:
    tank_info = pickle.load(f)
all_cylinders = tank_info['tanks'][tank_name]


In [25]:
in_cyls, out_cyls = multiway_filter_point_clouds(all_cylinders, radius_offset_max=0.5)

In [26]:
for c in in_cyls:
    c.update_filepath(new_dir=data_dir, num_keep_level=1, file_name=f'{c.station.name}.pcd')

### Read point clouds for inlier cylinders

In [27]:
# frind = 7
# pcds_down, pcds_full = load_point_clouds([in_cyls[i] for i in range(frind, frind+2)], file_dir=data_dir, full_normal=True)
# in_cyls.pop(3)
pcds_down, pcds_full = load_point_clouds(in_cyls, file_dir=data_dir, full_normal=True)
run_all = len(pcds_down) > 3

D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_004.pcd
D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_005.pcd
D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_006.pcd
D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_007.pcd
D:/REGISTRATION DATA/External/JB-567 27ft EXT Leica/thin_slice\Tank409 Scan_Plains_Tank_409-_Setup_008.pcd


Rough alignment - Phase 1
Triangle matching

In [28]:
points = np.empty([1,4])
for i, p in enumerate(pcds_full):
    _pts = np.asarray(p.points)
    points = np.r_[
        points,
        np.c_[_pts, i * np.ones([_pts.shape[0],1])]
    ]
write_array2txt(
    os.path.join(data_dir, 'full_with_ids_phase_1.txt'),
    points)


In [29]:
# for p in pcds_full:
#     points = np.asarray(p.points)
#     points[:,2] = 0
#     p.points = o3d.utility.Vector3dVector(points)
#     points = np.asarray(p.normals)
#     points[:,2] = 0
#     p.normals = o3d.utility.Vector3dVector(points)

In [30]:
# p0 = pcds_full[0]

In [31]:
# np.asarray(p0.normals)

In [32]:
# sel_ind = range(0,2)
# pcds_down = pcds_down[:2]
# pcds_full = pcds_full[:2]

## Pose graph

In [33]:
print("Full registration ...")
max_correspondence_distance_coarse = 0.5#voxel_size * 15
max_correspondence_distance_fine = 0.005#voxel_size * 1.5
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    pose_graph = full_registration(pcds_full,
                                   max_correspondence_distance_coarse,
                                   max_correspondence_distance_fine,
                                   num_overlap=1, method=phase)

Full registration ...
----SOURCE 0----
===== source_id=0, target_id=1 =====
Coarse alignment
Apply point-to-plane ICP
[Open3D DEBUG] ICP Iteration #0: Fitness 0.4305, RMSE 0.1456
[Open3D DEBUG] Residual : 1.51e-03 (# of elements : 8359)
[Open3D DEBUG] ICP Iteration #1: Fitness 0.4327, RMSE 0.1433
[Open3D DEBUG] Residual : 1.09e-04 (# of elements : 8401)
[Open3D DEBUG] ICP Iteration #2: Fitness 0.4317, RMSE 0.1477
[Open3D DEBUG] Residual : 1.04e-04 (# of elements : 8383)
[Open3D DEBUG] ICP Iteration #3: Fitness 0.4317, RMSE 0.1477
[Open3D DEBUG] Residual : 1.04e-04 (# of elements : 8383)
[Open3D DEBUG] ICP Iteration #4: Fitness 0.4317, RMSE 0.1480
[Open3D DEBUG] Residual : 1.04e-04 (# of elements : 8383)
[Open3D DEBUG] ICP Iteration #5: Fitness 0.4317, RMSE 0.1480
[Open3D DEBUG] Residual : 1.04e-04 (# of elements : 8383)
[Open3D DEBUG] ICP Iteration #6: Fitness 0.4317, RMSE 0.1480
[Open3D DEBUG] Residual : 1.04e-04 (# of elements : 8383)
Fine alignment
[Open3D DEBUG] ICP Iteration #0: F

Open3D uses the function `global_optimization` to perform pose graph optimization. Two types of optimization methods can be chosen: `GlobalOptimizationGaussNewton` or `GlobalOptimizationLevenbergMarquardt`. The latter is recommended since it has better convergence property. Class `GlobalOptimizationConvergenceCriteria` can be used to set the maximum number of iterations and various optimization parameters.

Class `GlobalOptimizationOption` defines a couple of options. `max_correspondence_distance` decides the correspondence threshold. `edge_prune_threshold` is a threshold for pruning outlier edges. `reference_node` is the node id that is considered to be the global space.

In [34]:
print("Optimizing PoseGraph ...")
# max_correspondence_distance_fine = 0.05
option = o3d.pipelines.registration.GlobalOptimizationOption(
    max_correspondence_distance=max_correspondence_distance_fine,
    edge_prune_threshold=0.25,
    reference_node=0)
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    o3d.pipelines.registration.global_optimization(
        pose_graph,
        o3d.pipelines.registration.GlobalOptimizationLevenbergMarquardt(),
        o3d.pipelines.registration.GlobalOptimizationConvergenceCriteria(),
        option)

Optimizing PoseGraph ...
[Open3D DEBUG] Validating PoseGraph - finished.
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 5 nodes and 4 edges.
[Open3D DEBUG] Line process weight : 0.046138
[Open3D DEBUG] [Initial     ] residual : 1.815094e-28, lambda : 1.092588e+01
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] [GlobalOptimizationLM] Optimizing PoseGraph having 5 nodes and 4 edges.
[Open3D DEBUG] Line process weight : 0.046138
[Open3D DEBUG] [Initial     ] residual : 1.815094e-28, lambda : 1.092588e+01
[Open3D DEBUG] Maximum coefficient of right term < 1.000000e-06
[Open3D DEBUG] CompensateReferencePoseGraphNode : reference : 0


The global optimization performs twice on the pose graph. The first pass optimizes poses for the original pose graph taking all edges into account and does its best to distinguish false alignments among uncertain edges. These false alignments have small line process weights, and they are pruned after the first pass. The second pass runs without them and produces a tight global alignment. In this example, all the edges are considered as true alignments, hence the second pass terminates immediately.

## Visualize optimization
The transformed point clouds are listed and visualized using `draw_geometries`.

In [35]:
# print("Transform points and display")
# for point_id in range(len(pcds_down)):
#     pcds_down[point_id].transform(pose_graph.nodes[point_id].pose)
# o3d.visualization.draw_geometries(pcds_down,
#                                   zoom=0.3412,
#                                   front=[0.4257, -0.2125, -0.8795],
#                                   lookat=[2.6172, 2.0475, 1.532],
#                                   up=[-0.0694, -0.9768, 0.2024])

## Make a combined point cloud
`PointCloud` has a convenience operator `+` that can merge two point clouds into a single one. In the code below, the points are uniformly resampled using `voxel_down_sample` after merging. This is recommended post-processing after merging point clouds since it can relieve duplicated or over-densified points.

In [36]:
transform_2D = []
for node in pose_graph.nodes:
    transform_matrix = node.pose
    transform_2D.append({
        'rotation': transform_matrix[:3,:3],
        'translation': transform_matrix[:3,-1].reshape([3,1])
    })
with open(os.path.join(data_dir, 'transform_2d.pkl'), 'wb') as f:
    pickle.dump(transform_2D, f)


Rough alignment - Phase 2
2D registration

In [37]:
inliner_pcd_combined = combine_point_cloud(pcds_full, pose_graph)
if run_all:
    write_array2txt(
        os.path.join(data_dir, 'full_with_ids_phase_2.txt'),
        inliner_pcd_combined)
else:
    write_array2txt(
        os.path.join(data_dir, f'full_{frind+1}_{frind+2}_with_ids.txt'),
        inliner_pcd_combined)
